In [12]:
import os
os.environ["MLFLOW_TRACKING_URI"] = "sqlite:///mlflow.db"  # must be set before any LazyClassifier() is created

import pandas as pd
import mlflow
from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTENC
from lazypredict.Supervised import LazyClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [13]:
TARGET_ROWS = 30000
TOLERANCE = 0.15  # skip any single repo that alone would push total too far past target

df = pd.read_csv("../data/processed/pr_snapshots_clean_v2.csv")

repo_sizes = df.groupby("repo_key").size().sample(frac=1, random_state=42)
keep_repos, total = [], 0
for repo, size in repo_sizes.items():
    if total >= TARGET_ROWS:
        break
    if total + size > TARGET_ROWS * (1 + TOLERANCE):
        continue
    keep_repos.append(repo)
    total += size

df = df[df.repo_key.isin(keep_repos)].copy()
print(f"{len(keep_repos)} repos, {len(df)} rows")

y = df.pop("merged_before_next")
X = df.select_dtypes(include="number").drop(columns=["pr_id", "number", "timeline_data_available"], errors="ignore")
print(f"shape: {X.shape}, target distribution: {y.value_counts().to_dict()}")

5 repos, 30867 rows
shape: (30867, 24), target distribution: {0: 19349, 1: 11518}


In [14]:
# group split: whole repos held out, never split across train/test
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=df["repo_key"]))
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
print(f"train repos: {df.repo_key.iloc[train_idx].nunique()}, test repos: {df.repo_key.iloc[test_idx].nunique()}")

train repos: 4, test repos: 1


In [21]:
# run 1: SMOTE, all ~30 models. SMOTE needs no NaN and can't see test data -- impute + resample train only.
# SMOTENC (not plain SMOTE) so binary/checkpoint-style columns don't get interpolated into fractions.
imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)

categorical_cols = ["checkpoint_day", "created_dow", "created_hour", "reviewer_assigned",
                    "has_response_yet", "is_first_time_contributor"]
cat_idx = [X_train_imp.columns.get_loc(c) for c in categorical_cols if c in X_train_imp.columns]

smote = SMOTENC(categorical_features=cat_idx, random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_imp, y_train)
print(f"before: {y_train.value_counts().to_dict()}, after SMOTE: {y_train_res.value_counts().to_dict()}")

mlflow.set_experiment("pr_bottleneck_smote")

# --- FIX STARTS HERE ---
# 1. End any currently active/dangling run
if mlflow.active_run():
    mlflow.end_run()

# 2. Temporarily disable autologging to prevent LazyClassifier loop crashes
mlflow.autolog(disable=True) 
# -----------------------

clf_smote = LazyClassifier(verbose=0, ignore_warnings=False, predictions=False)
models_smote, _ = clf_smote.fit(X_train_res, X_test, y_train_res, y_test)

# (Optional) Re-enable autologging for the rest of your notebook
mlflow.autolog(disable=False)

models_smote

2026/07/17 14:23:43 WARNING mlflow.sklearn: Failed to infer model signature: the trained model does not have a `predict` or `transform` function, which is required in order to infer the signature
2026/07/17 14:23:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/07/17 14:23:43 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2026/07/17 14:23:49 WARNING mlflow.sklearn: Training metrics will not be recorded because training labels were not specified. To automatically record training metrics, provide training labels as inputs to the model training function.
2026/07/17 14:23:54 INFO mlflow.tracking.fluent: Aut

before: {0: 16193, 1: 9805}, after SMOTE: {0: 16193, 1: 16193}


2026/07/17 14:23:54 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.
2026/07/17 14:23:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
C:\Users\VEDANG BARMAN\Desktop\Git_Pr_prediction\venv\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(
2026/07/17 14:24:06 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\VEDANG BARMAN\Desktop\Git_Pr_prediction\venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer colu

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Time Taken
Model,,,,,,,
XGBClassifier,0.628671,0.648167,0.688993,0.636776,0.681084,0.628671,10.113117
LGBMClassifier,0.637708,0.640191,0.687225,0.645522,0.669834,0.637708,9.632716
AdaBoostClassifier,0.619018,0.622838,0.660175,0.627421,0.654648,0.619018,12.161321
RandomForestClassifier,0.578558,0.617785,0.658864,0.583823,0.662050,0.578558,19.522803
Perceptron,0.544260,0.614416,0.672216,0.535494,0.683853,0.544260,7.772358
PassiveAggressiveClassifier,0.512426,0.603872,0.622712,0.485050,0.703528,0.512426,21.882203
LogisticRegression,0.581844,0.602570,0.622380,0.590474,0.640272,0.581844,9.478290
NuSVC,0.581639,0.594938,0.610939,0.590868,0.631473,0.581639,445.597949
LinearSVC,0.580407,0.594255,0.617691,0.589626,0.630973,0.580407,9.172485


In [16]:
mlflow.end_run()

In [23]:
# Sum up the total time taken by all models in seconds
total_seconds = models_smote['Time Taken'].sum()

# Convert it to minutes and seconds for better readability
minutes = int(total_seconds // 60)
seconds = int(total_seconds % 60)

print(f"Total LazyPredict execution time: {total_seconds:.2f} seconds")
print(f"Formatted time: {minutes} minutes and {seconds} seconds")

Total LazyPredict execution time: 2475.24 seconds
Formatted time: 41 minutes and 15 seconds


In [24]:
models_smote

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Time Taken
Model,,,,,,,
XGBClassifier,0.628671,0.648167,0.688993,0.636776,0.681084,0.628671,10.113117
LGBMClassifier,0.637708,0.640191,0.687225,0.645522,0.669834,0.637708,9.632716
AdaBoostClassifier,0.619018,0.622838,0.660175,0.627421,0.654648,0.619018,12.161321
RandomForestClassifier,0.578558,0.617785,0.658864,0.583823,0.662050,0.578558,19.522803
Perceptron,0.544260,0.614416,0.672216,0.535494,0.683853,0.544260,7.772358
PassiveAggressiveClassifier,0.512426,0.603872,0.622712,0.485050,0.703528,0.512426,21.882203
LogisticRegression,0.581844,0.602570,0.622380,0.590474,0.640272,0.581844,9.478290
NuSVC,0.581639,0.594938,0.610939,0.590868,0.631473,0.581639,445.597949
LinearSVC,0.580407,0.594255,0.617691,0.589626,0.630973,0.580407,9.172485
